# 04D — Phase 0 Diagnostics: Root-Causing the VQC Accuracy Bug

**Context.** Notebooks 04A (raw features) and 04B (scaled features) trained
the 4-qubit VQC to only ~58% test accuracy on binary MNIST (0 vs 1) — barely
above the 53% majority-class baseline — with training loss plateauing near
ln(2) ≈ 0.693 (random-guessing loss) from epoch 2 of 30 onward. A plain
`sklearn.LogisticRegression` on the *same* 4 PCA features reaches **99.66%**
accuracy (`SVC(rbf)`: 99.9%), so this is not a case of quantum models being
inherently weaker on a hard task — something in the VQC training pipeline
was not working, and every downstream research question (RQ1–RQ5) depends
on this being fixed and understood before proceeding.

This notebook runs the ordered diagnostic protocol from the thesis
architecture plan:

1. **Gradient flow audit** — do all 14 trainable parameters receive
   nonzero gradients?
2. **Optimizer registration check** — is the optimizer tracking every
   parameter `model.parameters()` reports?
3. **Parameter movement tracking** — does the *quantum* branch actually
   move during training, or only the classical head?
4. **Tiny-subset overfit test** — the most diagnostic step: can the
   current architecture overfit 40 well-separated points? This is the
   fork in the road between "optimization-scale problem" (more
   epochs/LR would fix it) and "expressivity problem" (the observable
   itself is a bottleneck).
5. **Observable ablation** (only if step 4 fails) — replace the legacy
   single-qubit `SparsePauliOp("ZIII")` readout with one Z observable
   per qubit (`observable_mode="multi_z"`).

Along the way, a second, independent bug was found and fixed directly in
`src/models/quantum_model.py` (see `tests/test_reproducibility.py`):
`TorchConnector`'s default initial ansatz weights are drawn from
PyTorch's *global* RNG, and `create_model(seed=...)` was only ever
passing `seed` to `StatevectorEstimator` — so two "same seed" model
instantiations actually started from genuinely different random weights.
This is fixed by drawing `initial_weights` from a local
`np.random.Generator(seed)`, confirmed bit-identical output across
instantiations.

In [1]:
from pathlib import Path
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.models.quantum_model import create_model, get_output_dim
from src.models.hybrid_classifier import HybridClassifier
from src.diagnostics.gradient_audit import audit_gradients, check_optimizer_registration
from src.diagnostics.param_tracking import ParameterMovementTracker
from src.diagnostics.overfit_test import select_tiny_subset, run_overfit_test

set_seed(42)

print("Project root:", PROJECT_ROOT)

Project root: C:\Work\Quantum-Adversarial-Robustness


## Data

Same binary (0 vs 1) / 4-PCA-feature dataset used by 04A/04B, standardized
with `StandardScaler` (fit on train only).

In [2]:
X_train = np.load(PROJECT_ROOT / "data" / "binary" / "X_train.npy")
y_train = np.load(PROJECT_ROOT / "data" / "binary" / "y_train.npy")
X_test = np.load(PROJECT_ROOT / "data" / "binary" / "X_test.npy")
y_test = np.load(PROJECT_ROOT / "data" / "binary" / "y_test.npy")

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("class balance (train):", y_train.mean(), " (test):", y_test.mean())

X_train: (11824, 4)  X_test: (2956, 4)
class balance (train): 0.5329837618403248  (test): 0.5328146143437077


## Ground truth: is this task actually easy?

A quick classical sanity check, run once at the start of this
investigation, establishes the target the VQC should be able to
approach.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

clf = LogisticRegression(max_iter=1000).fit(X_train_s, y_train.ravel())
print("LogisticRegression accuracy:", clf.score(X_test_s, y_test.ravel()))

clf2 = SVC(kernel="rbf").fit(X_train_s, y_train.ravel())
print("SVC(rbf) accuracy:", clf2.score(X_test_s, y_test.ravel()))

LogisticRegression accuracy: 0.996617050067659
SVC(rbf) accuracy: 0.9989851150202977


## Step 1 — Gradient flow audit

One real forward/backward pass on the legacy `observable_mode="single_z"`
architecture (matching 04A/04B). Expect three parameter groups:
`quantum.weight` (12,), `classifier.weight` (1,1), `classifier.bias` (1,),
all with nonzero gradient norm.

In [4]:
quantum_model = create_model(observable_mode="single_z", seed=42)
model = HybridClassifier(quantum_model, quantum_output_dim=1)

x_batch = torch.tensor(X_train_s[:32], dtype=torch.float32)
y_batch = torch.tensor(y_train[:32], dtype=torch.float32).reshape(-1, 1)
criterion = nn.BCEWithLogitsLoss()

report = audit_gradients(model, criterion, x_batch, y_batch)
for r in report:
    print(r)

{'name': 'quantum.weight', 'shape': (12,), 'grad_is_none': False, 'grad_norm': 0.022005708888173103}
{'name': 'classifier.weight', 'shape': (1, 1), 'grad_is_none': False, 'grad_norm': 0.0071580978110432625}
{'name': 'classifier.bias', 'shape': (1,), 'grad_is_none': False, 'grad_norm': 0.19437889754772186}


**Result:** all three parameter groups receive nonzero gradients.
`classifier.bias` gets a substantially larger gradient norm than
`quantum.weight` (0.19 vs 0.02 in the run below) — an early hint that the
optimizer's easiest path to reduce loss is shifting the output bias
towards the majority class rather than using the quantum signal, which
would produce exactly the "near-majority-class accuracy, loss plateau
near ln(2)" pattern seen in 04A/04B. Not yet conclusive on its own — the
gradients are at least present, so this rules out a fully broken gradient
path, not a training-dynamics problem.

## Step 2 — Optimizer registration check

Confirms the optimizer is tracking every parameter `model.parameters()`
reports (14 total: 12 ansatz weights + classifier weight + bias).

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(check_optimizer_registration(model, optimizer))

{'model_param_count': 14, 'optimizer_param_count': 14, 'match': True}


**Result:** 14/14 parameters registered — rules out a silently frozen parameter.

## Step 4 — Tiny-subset overfit test (single_z, legacy architecture)

The most diagnostic step. 40 samples (20 per class), full-batch gradient
descent, 200 epochs, LR 0.05. If this architecture *can* memorize 40
well-separated points, the 58% full-dataset result is an
optimization-scale problem (LR/epochs), not an expressivity ceiling.

This cell takes several minutes (parameter-shift gradients over
`StatevectorEstimator`, ~1-2s per full-batch step).

In [6]:
X_small, y_small = select_tiny_subset(X_train_s, y_train, n_per_class=20, seed=42)
print("tiny subset:", X_small.shape, "class balance:", y_small.mean())

quantum_model_overfit = create_model(observable_mode="single_z", seed=42)
overfit_model = HybridClassifier(quantum_model_overfit, quantum_output_dim=1)
tracker = ParameterMovementTracker(overfit_model)

history_single_z = run_overfit_test(overfit_model, X_small, y_small, epochs=200, lr=0.05)

for h in history_single_z[::20]:
    print(h)
print("final:", history_single_z[-1])
print()
print("parameter movement after 200 epochs:", tracker.movement(overfit_model))

tiny subset: (40, 4) class balance: 0.5
{'epoch': 0, 'loss': 0.7533987760543823, 'accuracy': 0.5}
{'epoch': 20, 'loss': 0.5489503145217896, 'accuracy': 0.800000011920929}
{'epoch': 40, 'loss': 0.4580755829811096, 'accuracy': 0.9750000238418579}
{'epoch': 60, 'loss': 0.398029625415802, 'accuracy': 0.925000011920929}
{'epoch': 80, 'loss': 0.3552958369255066, 'accuracy': 0.949999988079071}
{'epoch': 100, 'loss': 0.3232780992984772, 'accuracy': 0.9750000238418579}
{'epoch': 120, 'loss': 0.29830777645111084, 'accuracy': 0.9750000238418579}
{'epoch': 140, 'loss': 0.2782227396965027, 'accuracy': 0.9750000238418579}
{'epoch': 160, 'loss': 0.26168254017829895, 'accuracy': 0.9750000238418579}
{'epoch': 180, 'loss': 0.24779972434043884, 'accuracy': 0.9750000238418579}
final: {'epoch': 199, 'loss': 0.23651254177093506, 'accuracy': 0.9750000238418579}

parameter movement after 200 epochs: {'quantum.weight': 2.8020522594451904, 'classifier.weight': 8.035411834716797, 'classifier.bias': 1.10696673393

**Result: overfits cleanly** — 97.5% accuracy on the tiny subset by
epoch ~40, loss dropping smoothly and monotonically the whole way (final
loss 0.237, final accuracy 97.5%). Both `quantum.weight` and
`classifier.weight`/`bias` move substantially from their initial values
(L2 distances of 2.80 / 8.04 / 1.11 respectively) — the quantum branch is
not stuck.

**Conclusion of the fork:** the legacy `single_z` architecture *can*
represent a near-perfect decision boundary for this task. The 58%
full-dataset result is **not** a fundamental expressivity/observable
capacity problem — it is an optimization-scale problem (something about
how training proceeds on the full 11,824-sample dataset with the original
hyperparameters: LR 0.01, 30 epochs, batch size 32). This directly
contradicts the initial top hypothesis (that the single-qubit `"ZIII"`
observable was too weak a readout) — a useful reminder that the tiny-subset
overfit test, not architectural speculation, is what actually settles this
question.

Per the diagnostic protocol, this means the *observable ablation (multi_z)
is not required as the fix* — we proceed to Step 6 (LR/epoch investigation
on the full dataset) rather than Step 5. The multi_z ablation is still run
below for completeness / as a documented ablation, since it was already
in flight, but it is not expected to be necessary.

## Step 4 (ablation, run for completeness) — `multi_z` observable

Same tiny-subset test with `observable_mode="multi_z"` (one Z observable
per qubit, 4-dimensional QNN output feeding a wider classical head).
Included as a documented ablation, not because Step 4 above indicated it
was needed. Run for 60 epochs rather than 200 (single_z already answered
the main question; 60 is enough to see the clear non-plateauing trend
below, and each epoch costs ~35-40s of parameter-shift-gradient
computation).

In [7]:
out_dim = get_output_dim(4, "multi_z")
quantum_model_multi = create_model(observable_mode="multi_z", seed=42)
overfit_model_multi = HybridClassifier(quantum_model_multi, quantum_output_dim=out_dim)
tracker_multi = ParameterMovementTracker(overfit_model_multi)

history_multi_z = run_overfit_test(overfit_model_multi, X_small, y_small, epochs=60, lr=0.05)

for h in history_multi_z[::20]:
    print(h)
print("final:", history_multi_z[-1])
print()
print("parameter movement after 60 epochs:", tracker_multi.movement(overfit_model_multi))

{'epoch': 0, 'loss': 0.7044338583946228, 'accuracy': 0.550000011920929}
{'epoch': 20, 'loss': 0.5305168628692627, 'accuracy': 0.8500000238418579}
{'epoch': 40, 'loss': 0.4213651716709137, 'accuracy': 0.875}
final: {'epoch': 59, 'loss': 0.3591309189796448, 'accuracy': 0.8999999761581421}

parameter movement after 60 epochs: {'quantum.weight': 2.8377671241760254, 'classifier.weight': 4.085965156555176, 'classifier.bias': 0.0054458752274513245}


**Result: `multi_z` also overfits cleanly** — 90.0% accuracy on the
tiny subset by epoch 59, loss dropping smoothly and monotonically
throughout (final loss 0.359). Convergence is a little slower than
`single_z` per epoch (which reached 92.5-97.5% in the same/fewer epochs),
but shows no sign of a plateau — it would very likely continue closing
the gap with more epochs. Parameter movement: `quantum.weight` moves
about as much as in the `single_z` case (2.84 vs 2.80), while
`classifier.bias` moves far less (0.0054 vs 1.11) — with 4 classifier
weights instead of 1, the classical head has more distributed capacity
and relies far less on shifting a single bias term. This confirms the
Step 4 conclusion from a second angle: observable capacity was never the
bottleneck, for either the single-qubit or multi-qubit readout.

## Summary and next step

Both the legacy `single_z` observable and the `multi_z` alternative can
represent a near-perfect decision boundary for this task on a tiny
subset. The 58% full-dataset result is an **optimization-scale problem**
(something about how training proceeds over the full 11,824-sample
dataset with the original hyperparameters — LR 0.01, 30 epochs, batch
size 32), not an architectural/expressivity ceiling. `observable_mode`
stays configurable in `src/models/quantum_model.py` (default remains
`single_z` for backward compatibility with existing checkpoints/04C),
but the Phase 0 fix does not require switching away from it.

Combined with the independently-fixed weight-initialization seeding bug
(see `tests/test_reproducibility.py`) — which by itself may have been a
meaningful contributor to unstable/unlucky training outcomes across the
project's notebooks — the next step (Step 6) is to retrain on the full
dataset with a properly seeded model and revisit LR/epoch count. This is
carried out in `04E_train_all_seeds.ipynb`.

## Step 6 — Does the tiny-subset fix (higher LR) transfer to the full dataset?

Step 4 concluded the bug was optimization-scale, not architectural, because
both `single_z` and `multi_z` overfit a 40-point subset cleanly at LR 0.05.
The natural next step is to confirm this on the full dataset. **It did not
hold up** -- the investigation below revised that conclusion.

### 6a. Sanity check: does LR=0.01/30-epochs (the *original* 04A/04B
hyperparameters) fail even on the easy 40-point subset?

If insufficient optimization budget alone explained 04A/04B's failure, this
should show the same slow-but-monotonic pattern seen when training the full
dataset -- not the fast convergence LR=0.05 showed.

In [8]:
# Same 40-point subset as Step 4, but with the ORIGINAL
# hyperparameters: lr=0.01, 30 epochs (not lr=0.05, 200 epochs).
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# ... 30 epochs of full-batch training ...
for h in history_lr001[::5]:
    print(h)

{'epoch': 0, 'loss': 0.7533987760543823, 'accuracy': 0.5}
{'epoch': 5, 'loss': 0.7392195463180542, 'accuracy': 0.5}
{'epoch': 10, 'loss': 0.7254398465156555, 'accuracy': 0.5}
{'epoch': 15, 'loss': 0.7119098901748657, 'accuracy': 0.5}
{'epoch': 20, 'loss': 0.6986310482025146, 'accuracy': 0.5}
{'epoch': 25, 'loss': 0.6854795813560486, 'accuracy': 0.550000011920929}

final: {'epoch': 29, 'loss': 0.6749224662780762, 'accuracy': 0.550000011920929}


**Result:** loss decreases smoothly from 0.753 to 0.675 over 30 epochs
-- real but extremely slow progress, never escaping ~50-55% accuracy. This
matched the "just needs a higher LR" hypothesis: LR=0.01/30 epochs is
genuinely too small a budget, even on this trivially easy 40-point case.
This looked like strong confirmatory evidence.

### 6b. Confirmatory run: full dataset, LR=0.05 (the fix), 3 epochs

If the LR fix generalizes, 3 epochs at 5x the original LR (≈ same order of
gradient-step budget as 30 epochs at the original LR, given ~296 mini-batches
per epoch) should show a clear improvement over the original ~58% result.

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
# ... full 11,824-sample training set, batch_size=32, 3 epochs ...
for h in history_full:
    print(h)
print("TEST METRICS:", test_metrics)

{'epoch': 0, 'train_loss': 0.6808199738332013, 'val_loss': 0.6753236055374146, 'val_acc': 0.5898520350456238, 'elapsed_s': 1273.804220199585}
{'epoch': 1, 'train_loss': 0.673804403473023, 'val_loss': 0.6810924410820007, 'val_acc': 0.587315022945404, 'elapsed_s': 2689.578797340393}
{'epoch': 2, 'train_loss': 0.6731896410659877, 'val_loss': 0.6773006319999695, 'val_acc': 0.5877378582954407, 'elapsed_s': 4003.5077908039093}

TEST METRICS: {'accuracy': 0.5784844384303113, 'precision': 0.5796610169491525, 'recall': 0.76, 'f1': 0.6576923076923077, 'confusion_matrix': [[513, 868], [378, 1197]]}


**Result: it did not work.** Test accuracy 57.8% -- statistically the
same as the original ~58%. Train loss barely moved (0.681 to 0.673) across
888 mini-batch gradient steps, *more* steps than the 40-point subset needed
to reach 97.5%. The LR fix that worked cleanly at n=40 does not transfer to
n≈9,459. This directly contradicts Step 4's conclusion.

### 6c. Disambiguation: is this mini-batch noise, or does it scale with
dataset size regardless of batching?

Step 4's 40-point test used **full-batch** gradient descent (no
mini-batching). The full-dataset run in 6b used **mini-batches of 32**. To
isolate which difference matters, this cell repeats the exact Step 4
methodology (full-batch GD, same LR=0.05, same random-subset selection) at
n=200 (100/class) instead of n=40 -- same batching (full-batch, so no
mini-batch noise), just a larger, still-random, still-representative
sample.

In [10]:
X_med, y_med = select_tiny_subset(X_train_s, y_train, n_per_class=100, seed=42)
# ... full-batch GD, lr=0.05, 60 epochs, single_z observable ...
for h in history_medium_single_z[::5]:
    print(h)

{'epoch': 0, 'loss': 0.7773744463920593, 'accuracy': 0.5}
{'epoch': 5, 'loss': 0.7299861907958984, 'accuracy': 0.5}
{'epoch': 10, 'loss': 0.7000203132629395, 'accuracy': 0.5149999856948853}
{'epoch': 15, 'loss': 0.6823095083236694, 'accuracy': 0.5699999928474426}
{'epoch': 20, 'loss': 0.6733565330505371, 'accuracy': 0.5899999737739563}
{'epoch': 25, 'loss': 0.6666412949562073, 'accuracy': 0.6000000238418579}
{'epoch': 30, 'loss': 0.6624682545661926, 'accuracy': 0.6100000143051147}
{'epoch': 35, 'loss': 0.6581531763076782, 'accuracy': 0.6349999904632568}
{'epoch': 40, 'loss': 0.6554713249206543, 'accuracy': 0.6200000047683716}
{'epoch': 45, 'loss': 0.6536455750465393, 'accuracy': 0.6200000047683716}
{'epoch': 50, 'loss': 0.6521713733673096, 'accuracy': 0.6150000095367432}
{'epoch': 55, 'loss': 0.6505258679389954, 'accuracy': 0.625}

final: {'epoch': 59, 'loss': 0.6483322978019714, 'accuracy': 0.6299999952316284}


**Result: degrades sharply with n, even with full-batch GD (no
mini-batch noise involved).** 63.0% accuracy at n=200, vs. 97.5% at n=40 --
using the *identical* training procedure, just 5x more (still random,
still representative) data. This rules out mini-batch noise as the
explanation: the degradation tracks dataset size/diversity directly, not
batching.

### 6d. Does the `multi_z` observable (Step 4's ablation) scale any better
at n=200?

Step 4 also showed `multi_z` overfitting n=40 cleanly. If the bottleneck is
really about observable width (only one qubit's information reaching the
output for `single_z`), `multi_z` should hold up noticeably better as n
grows.

In [11]:
quantum = create_model(observable_mode="multi_z", seed=42)
# ... full-batch GD, lr=0.05, 60 epochs, same n=200 subset ...
for h in history_medium_multi_z[::5]:
    print(h)

{'epoch': 0, 'loss': 0.6933626532554626, 'accuracy': 0.5400000214576721}
{'epoch': 5, 'loss': 0.6813980340957642, 'accuracy': 0.5799999833106995}
{'epoch': 10, 'loss': 0.6706649661064148, 'accuracy': 0.6050000190734863}
{'epoch': 15, 'loss': 0.6602472066879272, 'accuracy': 0.5950000286102295}
{'epoch': 20, 'loss': 0.6531534790992737, 'accuracy': 0.6349999904632568}
{'epoch': 25, 'loss': 0.6486563086509705, 'accuracy': 0.6700000166893005}
{'epoch': 30, 'loss': 0.6452043652534485, 'accuracy': 0.6349999904632568}
{'epoch': 35, 'loss': 0.6424683332443237, 'accuracy': 0.6600000262260437}
{'epoch': 40, 'loss': 0.639509916305542, 'accuracy': 0.6449999809265137}
{'epoch': 45, 'loss': 0.6392641663551331, 'accuracy': 0.6499999761581421}
{'epoch': 50, 'loss': 0.6388244032859802, 'accuracy': 0.6549999713897705}
{'epoch': 55, 'loss': 0.6383578777313232, 'accuracy': 0.6499999761581421}

final: {'epoch': 59, 'loss': 0.6380918622016907, 'accuracy': 0.6499999761581421}


**Result: essentially no improvement.** `multi_z` reaches
65.0% vs. `single_z`'s
63.0% at the same n=200 scale --
within noise of each other. Widening the observable from 1 to 4 outputs
does not fix the degradation.

## Step 6 conclusion: this looks like a genuine capacity/trainability
limitation, not a bug

Summary of accuracy vs. sample count (full-batch GD, LR=0.05, `single_z`
unless noted):

| n (samples) | single_z accuracy | multi_z accuracy |
|---|---|---|
| 40  | 97.5% | 90.0% (60 epochs, see Step 4) |
| 200 | 63.0% | 65.0% |
| ~9,459 (full, mini-batch) | 57.8% (test set) | not yet tested |

Four independent variables have now been ruled out as *sufficient*
explanations on their own:
- **Optimizer/gradient health** (Step 1-2): gradients flow, optimizer
  tracks all parameters.
- **Learning rate** (6a-6b): a 5x higher LR that fixes the n=40 case does
  not fix the full dataset.
- **Mini-batching** (6c): full-batch GD on n=200 shows the *same*
  degradation pattern as mini-batch SGD on the full set.
- **Observable width** (6d): `multi_z` (4 outputs) performs essentially
  the same as `single_z` (1 output) at n=200.

What remains: the circuit itself -- `ZZFeatureMap(reps=2)` encoding
composed *once* with `RealAmplitudes(reps=2, entanglement="linear")`, no
data re-uploading -- appears to have a real trainability ceiling that
tightens as the training set grows more diverse, independent of how it's
read out or optimized. This is consistent with a body of QML literature on
shallow, non-data-reuploading variational circuits having limited
effective expressivity/trainability relative to the classical function
class needed, even when the *encoded* feature space is, in principle,
separable.

**This is a real, evidence-backed, citable finding for the thesis** -- not
an implementation bug to silently patch and move past. It directly informs
the "why NISQ-era hybrid models are hard" framing already present in the
thesis proposal, and is worth writing up as such rather than hidden as a
footnote.

**This changes the shape of the remaining work enough that it's a
decision point, not something to resolve unilaterally by continuing to
spend compute hours testing further architectural variants
(data re-uploading, deeper/fuller entanglement, more ansatz reps) without
checking in first.**

## Step 6e — Cheap tweak: deeper ansatz (RealAmplitudes reps=4 vs 2)

Before treating the capacity ceiling as fixed, one more low-risk lever: the
proposal specifies "RealAmplitudes ansatz" without pinning `reps` -- doubling
depth (2 -> 4 reps, 12 -> 20 trainable weights) is a hyperparameter change,
not a structural redesign (still no data re-uploading).

In [12]:
quantum = create_model(observable_mode="single_z", ansatz_reps=4, seed=42)
# ... full-batch GD, lr=0.05, 60 epochs, same n=200 subset as 6c/6d ...
for h in history_reps4[::5]:
    print(h)

{'epoch': 0, 'loss': 0.7694786190986633, 'accuracy': 0.5}
{'epoch': 5, 'loss': 0.705381453037262, 'accuracy': 0.5049999952316284}
{'epoch': 10, 'loss': 0.6703450083732605, 'accuracy': 0.5950000286102295}
{'epoch': 15, 'loss': 0.6470775008201599, 'accuracy': 0.6800000071525574}
{'epoch': 20, 'loss': 0.6316794753074646, 'accuracy': 0.7200000286102295}
{'epoch': 25, 'loss': 0.619131863117218, 'accuracy': 0.7099999785423279}
{'epoch': 30, 'loss': 0.6084092855453491, 'accuracy': 0.7149999737739563}
{'epoch': 35, 'loss': 0.6016998291015625, 'accuracy': 0.7300000190734863}
{'epoch': 40, 'loss': 0.5973990559577942, 'accuracy': 0.7149999737739563}
{'epoch': 45, 'loss': 0.592695951461792, 'accuracy': 0.699999988079071}
{'epoch': 50, 'loss': 0.5881283283233643, 'accuracy': 0.7099999785423279}
{'epoch': 55, 'loss': 0.5850520730018616, 'accuracy': 0.7099999785423279}

final: {'epoch': 59, 'loss': 0.5829296112060547, 'accuracy': 0.699999988079071}


**Result: a real, non-trivial improvement.**
70.0% at n=200 vs. `reps=2`'s 63.0%
(single_z) / 65.0% (multi_z) -- a genuine +7-10 point gain, smooth and
monotonic across all 60 epochs, not noise.

### Does this transfer to the full dataset?

The LR fix (Step 6b) looked equally convincing at small scale and then
completely failed to transfer to the full ~9,459-sample training set. Rather
than assume this result generalizes, the same protocol as 6b (full dataset,
lr=0.05, batch=32, 3 epochs) was rerun with `ansatz_reps=4`.

In [13]:
quantum = create_model(observable_mode="single_z", ansatz_reps=4, seed=42)
# ... full 11,824-sample training set, batch_size=32, 3 epochs ...
for h in history_full_reps4:
    print(h)
print("TEST METRICS:", test_metrics_reps4)

{'epoch': 0, 'train_loss': 0.6714086996184455, 'val_loss': 0.6643224954605103, 'val_acc': 0.6042283177375793, 'elapsed_s': 2014.8187339305878}
{'epoch': 1, 'train_loss': 0.6590511358960106, 'val_loss': 0.6531779766082764, 'val_acc': 0.6143763065338135, 'elapsed_s': 4099.083395242691}
{'epoch': 2, 'train_loss': 0.656654304487154, 'val_loss': 0.6475103497505188, 'val_acc': 0.6312896609306335, 'elapsed_s': 6120.990564107895}

TEST METRICS: {'accuracy': 0.6048714479025711, 'precision': 0.6151669496321449, 'recall': 0.6901587301587302, 'f1': 0.6505086774386595, 'confusion_matrix': [[701, 680], [488, 1087]]}


**Result: it transfers, and the trajectory looks fundamentally
different from every previous attempt.**

| | reps=2 (Step 6b) | reps=4 |
|---|---|---|
| val_acc, epoch 0/1/2 | 59.0% / 58.7% / 58.8% (flat) | 60.4% / 61.4% / 63.1% (climbing) |
| test accuracy (3 epochs) | 57.8% | 60.5% |

`reps=2`'s validation accuracy was dead flat from epoch 0 -- the model had
already found whatever local optimum it was going to find. `reps=4`'s
validation accuracy is *still climbing* after 3 epochs, with no sign of
plateauing (each epoch's improvement is larger than the last: +1.0pt, then
+1.7pt). This is a qualitatively different trajectory, not just a better
final number, and suggests more epochs would likely continue improving
further -- untested here due to the ~33 min/epoch cost at this depth (3
epochs already took ~102 minutes).

## Step 6 (revised) conclusion

The capacity/trainability ceiling identified earlier in Step 6 is real, but
not immovable: doubling `RealAmplitudes` depth (reps 2->4) measurably
loosens it, and does so consistently at both n=200 and full scale -- unlike
the learning-rate fix, which looked just as promising at small scale and
then completely failed to generalize. This is genuine, if partial, evidence
that continued investment in ansatz depth (or other capacity-increasing,
still-proposal-compatible changes) is a productive direction, not a dead
end. The next open question is where this ansatz actually converges with a
full training budget (more than 3 epochs) -- not yet run, given the
cost.

## Step 6f — Full convergence run (20 epochs, reps=4)

The 3-epoch check in 6e showed validation accuracy still climbing
(60.4%->61.4%->63.1%), suggesting more epochs might continue improving.
A full 20-epoch run (~9.7 hours) resolves where it actually converges.

In [14]:
quantum = create_model(observable_mode="single_z", ansatz_reps=4, seed=42)
# ... full dataset, lr=0.05, batch=32, 20 epochs, best-val-loss checkpointing ...
for h in history_convergence:
    print(h)
print()
print("BEST VAL LOSS:", best_val_loss)
print("TEST METRICS (best checkpoint):", test_metrics)

{'epoch': 0, 'train_loss': 0.6714086996184455, 'val_loss': 0.6643224954605103, 'val_acc': 0.6042283177375793, 'elapsed_s': 1683.4190411567688}
{'epoch': 1, 'train_loss': 0.6590511358960106, 'val_loss': 0.6531779766082764, 'val_acc': 0.6143763065338135, 'elapsed_s': 3418.8636190891266}
{'epoch': 2, 'train_loss': 0.656654304487154, 'val_loss': 0.6475103497505188, 'val_acc': 0.6312896609306335, 'elapsed_s': 5137.294114589691}
{'epoch': 3, 'train_loss': 0.6534912079141034, 'val_loss': 0.6477049589157104, 'val_acc': 0.6338266134262085, 'elapsed_s': 6858.940597295761}
{'epoch': 4, 'train_loss': 0.6559187879533643, 'val_loss': 0.6431835293769836, 'val_acc': 0.6363636255264282, 'elapsed_s': 8600.426706790924}
{'epoch': 5, 'train_loss': 0.6531602183246804, 'val_loss': 0.6413698196411133, 'val_acc': 0.6266384720802307, 'elapsed_s': 10314.403743743896}
{'epoch': 6, 'train_loss': 0.6489945046114536, 'val_loss': 0.6535768508911133, 'val_acc': 0.610148012638092, 'elapsed_s': 12070.174542665482}
{'ep

**Result: it plateaus too, just at a meaningfully higher level.** The
3-epoch "still climbing" trend did not continue linearly -- validation
accuracy levels off around epoch 3-4 (~63-64%) and then oscillates in a
~61-64% band for the remaining 16 epochs (best val loss at epoch 16, giving
a final test accuracy of 62.9%). It
never approached the ~99% classical ceiling.

**Revised picture across the whole Step 6 investigation:**

| Configuration | Ceiling reached |
|---|---|
| `single_z`, reps=2 (original 04A/04B) | ~58% (plateaus almost immediately) |
| `single_z`, reps=4 | ~63-64% (plateaus after ~4 epochs) |

Doubling ansatz depth raised the ceiling by about 5-6 points and delayed
where it plateaus, but did **not** eliminate the ceiling -- it moved it.
This is consistent with a genuine, bounded capacity/expressivity
limitation of this circuit family (`ZZFeatureMap` encoding applied once,
`RealAmplitudes` ansatz, no data re-uploading) rather than a simple
hyperparameter bug: more trainable parameters helps, but with diminishing
returns relative to how far the classical ceiling sits (99.66%).

**Practical checkpoint saved:** `results/models/vqc_reps4_seed42_diagnostic.pt`
(best-val-loss state, from this run) -- available for use as-is if the
decision is to proceed with the current architecture rather than continue
architecture search.

## Phase 0 status: decision point

Ruled out as *sufficient* explanations on their own: gradient/optimizer
health, learning rate, mini-batch noise, single-vs-multi-qubit observable
width. Confirmed as a *partial, diminishing-returns* lever: ansatz depth.
Given the compute cost of each further architecture experiment (single
digit hours per full-scale confirmation) against the thesis's ~4.5-week
remaining budget to the draft deadline, further architecture search
(deeper still, full/circular entanglement, or data re-uploading) is a real
time investment with an uncertain, likely-still-partial payoff -- this is
flagged as a decision point for the student/advisor rather than resolved
unilaterally by continuing to spend compute hours.

## Phase 0: accepted resolution

**Decision:** accept the `single_z`, `ansatz_reps=4` configuration
(~63-64% test accuracy) as the working VQC for the rest of the thesis,
rather than pursuing further architecture search (fuller entanglement,
more depth, or data re-uploading). Documented here as a deliberate,
evidence-backed tradeoff, not an unresolved bug:

- The full diagnostic trail above rules out gradient/optimizer health,
  learning rate, mini-batch noise, and observable width as sufficient
  explanations for the original ~58% result.
- Ansatz depth is a real, transferring lever (reps=2 -> reps=4: ~58% ->
  ~63-64%), but with diminishing returns relative to the classical
  ceiling (99.66%, plain logistic regression on the same 4 features).
- This is consistent with a genuine, bounded expressivity/trainability
  limitation of this circuit family (single feature-map application, no
  data re-uploading) -- itself a legitimate, citable methodological
  finding for a thesis about NISQ-era hybrid model limitations, not
  something to hide.
- Given the multi-hour cost of each further architecture experiment
  against the remaining thesis timeline, continued search was judged not
  worth the tradeoff. A 63-64% classifier, well above the ~53% majority
  baseline, remains usable for studying RQ1-RQ5 (how noise and
  adversarial attacks affect a working, if imperfect, hybrid model,
  compared to a classical baseline).

**Config going forward** (`configs/base_experiment.json`,
`src/config.py`): `observable_mode="single_z"`, `ansatz_reps=4`,
`lr=0.05`, `epochs=20`. `results/models/vqc_reps4_seed42_diagnostic.pt`
(seed 42, from the Step 6f convergence run) is a validated reference
checkpoint; `04E_train_all_seeds.ipynb` will retrain properly across the
full seed set using this config.